# 📓 Notebook 3: RAG + LLM + Agent, all together 🏆

We have learned all the pieces:
* **Notebook 1**: PDF → chunks → embeddings → FAISS (the *search* part)
* **Notebook 2**: LLM → system prompt → memory → agent (the *brain* part)

Now let's put them together and run a **competition** between three chatbots, asking all three the **same questions**:

| Round | Chatbot | How it works |
|---|---|---|
| 1 | 🧠 **LLM alone** | just asks the LLM |
| 2 | 📚 **Classic RAG** | always: search the PDF → give the chunks to the LLM |
| 3 | 🤖 **Agentic RAG** | an agent that *decides* when to search, and *remembers* the chat |

Which one wins? 🥇

## 1. Installation 📦
Run this once. After a fresh install, restart the kernel: **Kernel → Restart**.

In [ ]:
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu "torch==2.14.0" "sentence-transformers==6.1.0" "langchain-huggingface==1.2.2" "langchain==1.4.2" "langchain-core==1.6.5" "langchain-community==0.4.2" "langchain-text-splitters==1.1.2" "langgraph==1.2.12" "langchain-google-genai==4.4.0" "langchain-groq==1.1.3" "faiss-cpu==1.15.1" "pypdf==6.19.0" "python-dotenv==1.2.3"

## 2. Import the libraries 📚

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv("../.env")   # loads GROQ_API_KEY and GEMINI_API_KEY
print("✅ All libraries loaded!")

## 3. Load the LLM 🧠

In [ ]:
PROVIDER = "groq"   # 👈 change to "gemini" to try Google Gemini

if PROVIDER == "groq":
    api_key = os.getenv("GROQ_API_KEY") or getpass("Paste your GROQ_API_KEY: ")
    llm = ChatGroq(model="llama-3.1-8b-instant", api_key=api_key, temperature=0)
else:
    api_key = os.getenv("GEMINI_API_KEY") or getpass("Paste your GEMINI_API_KEY: ")
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key, temperature=0)

print("✅ LLM ready:", PROVIDER)

## 4. Build the knowledge base 📚 (everything from Notebook 1, in one cell)

In [ ]:
pages = PyPDFLoader("../data/sample_syllabus.pdf").load()
print(f"📄 Loaded {len(pages)} pages")

chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150).split_documents(pages)
print(f"✂️  Split into {len(chunks)} chunks")

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embedding_model)
print(f"🔢 Stored {vector_db.index.ntotal} vectors of {vector_db.index.d} numbers each in FAISS")

### 🎯 The test questions
We ask every chatbot the same four messages. Notice that **message 3 is a follow-up**: "*that*" only makes sense if the chatbot remembers message 2.

In [ ]:
Q1 = "What must the mini project report include?"          # a fact that only exists in OUR PDF
Q2 = "How are the internal (CIE) marks divided?"            # another PDF fact
Q3 = "Which part of that has the most marks?"               # follow-up: needs memory!
Q4 = "Hi! I'm Anu. Thanks for the help!"                    # small talk: no need to search the PDF

---
## 🥊 Round 1: LLM alone 🧠
No PDF, no memory. Just the LLM and its general knowledge.

In [ ]:
for question in [Q1, Q2, Q3]:
    print("❓", question)
    print("🧠", llm.invoke(question).text)
    print("-" * 80)

🤔 **What happened?** The LLM has **never seen our PDF**. It either says it does not know or, worse, **invents** a plausible answer. Inventing is called a **hallucination**.

---
## 🥊 Round 2: Classic RAG 📚
A fixed pipeline that **always** does the same two steps:
1. 🔍 **Retrieve**: search FAISS for the 3 closest chunks
2. ✍️ **Generate**: put those chunks into the prompt and ask the LLM

We build the prompt with a LangChain **prompt template**. The `|` symbol pipes the prompt into the LLM, forming a **chain**.

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a teaching assistant. Answer using ONLY the context below. "
               "If the answer is not in the context, say you don't know.\n\nContext:\n{context}"),
    ("human", "{question}"),
])
rag_chain = rag_prompt | llm      # prompt ──► LLM


def classic_rag(question):
    results = vector_db.similarity_search_with_score(question, k=3)
    print("❓", question)
    print("🔍 Retrieved chunks:")
    for doc, distance in results:
        print(f"   page {doc.metadata['page'] + 1} · distance {distance:.3f} · {doc.page_content[:70]!r}...")
    context = "\n\n".join(doc.page_content for doc, _ in results)
    answer = rag_chain.invoke({"context": context, "question": question}).text
    print("📚", answer)
    print("-" * 80)


for question in [Q1, Q2, Q3, Q4]:
    classic_rag(question)

🤔 **What happened?**
* ✅ Q1 and Q2 are now answered **from the PDF**. RAG fixed the hallucination!
* ❌ Q3: the pipeline has **no memory**, so it searched for "*Which part of that has the most marks?*" on its own. What is "that"? It may grab the wrong chunk (e.g. the exam's Part B).
* ❌ Q4: it searched the PDF for "*Hi! I'm Anu*". That is wasted work, because the pipeline **always** searches.

---
## 🥊 Round 3: Agentic RAG 🤖
Now the PDF search becomes a **tool** that the agent **may** use, and the agent gets **memory**:
* `@tool`: the docstring tells the LLM *when* to use it
* `system_prompt`: the agent's rules
* `InMemorySaver`: the agent's memory, one conversation per `thread_id`

In [ ]:
@tool
def search_pdf(query: str) -> str:
    """Search the course handbook PDF. Use it for any question about the course, exams, marks or projects."""
    print(f"   🔧 Agent decided to search the PDF for: '{query}'")
    results = vector_db.similarity_search(query, k=3)
    return "\n\n".join(doc.page_content for doc in results)


agent = create_agent(
    model=llm,
    tools=[search_pdf],
    system_prompt=(
        "You are a friendly teaching assistant. Use the search_pdf tool for questions about the course. "
        "Answer only from what the tool returns. For greetings or small talk, just reply without searching. "
        "Keep answers short."
    ),
    checkpointer=InMemorySaver(),
)
memory_config = {"configurable": {"thread_id": "anu-chat"}}


def agentic_rag(question):
    print("❓", question)
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, memory_config)
    print("🤖", result["messages"][-1].text)
    print("-" * 80)


for question in [Q1, Q2, Q3, Q4]:
    agentic_rag(question)

🤔 **What happened?**
* ✅ Q1 and Q2: the agent **chose** to call `search_pdf` and answered from the PDF.
* ✅ Q3: thanks to **memory** it knows "*that*" means the CIE marks. It may even search with a better query that it wrote itself!
* ✅ Q4: **no search**. The agent decided a greeting does not need the PDF.

### 👀 Look inside the agent's memory
Every message, tool call and tool result is stored:

In [ ]:
saved = agent.get_state(memory_config).values["messages"]
print(f"🧠 The agent remembers {len(saved)} messages:\n")
for message in saved:
    kind = type(message).__name__
    if getattr(message, "tool_calls", None):
        text = f"(decided to call {message.tool_calls[0]['name']} with {message.tool_calls[0]['args']})"
    else:
        text = message.text[:90].replace("\n", " ")
    print(f"  {kind:<13} {text}")

One more memory test: the agent should remember **your name** from Q4.

In [ ]:
agentic_rag("What is my name, and what was the first thing I asked you?")

---
## 🏆 Scoreboard

| | 🧠 LLM alone | 📚 Classic RAG | 🤖 Agentic RAG |
|---|:---:|:---:|:---:|
| Knows what is in **our PDF** | ❌ | ✅ | ✅ |
| Avoids **hallucinations** | ❌ | ✅ | ✅ |
| **Remembers** the conversation | ❌ | ❌ | ✅ |
| **Decides** when to search | ❌ | ❌ (always searches) | ✅ |
| Can **improve its own search query** | ❌ | ❌ | ✅ |
| Simple and predictable | ✅ | ✅ | ⚠️ (the LLM makes the decisions) |

**Takeaway:**
* **RAG** gives an LLM *knowledge* it was never trained on.
* An **agent** adds *decisions* and *memory*.
* Together they make the chatbots you see in real products, including our web app (`backend/core_rag.py`), which uses this Round 3 design.

## 🧪 Try it yourself
* Switch `PROVIDER` to `"gemini"`. Does the scoreboard change?
* Add your own question Q5. Can you find one where Classic RAG beats the agent?
* Load **your own PDF** in step 4 and ask all three chatbots about it.
* Give the agent a second tool, e.g. `def todays_date() -> str`, and ask *"How many days until the exam?"*